# 02. 前処理: 欠損値の扱いと、カテゴリ変数の渡し方

**このノートブックの目的:** モデルに渡す前の処理(欠損値・カテゴリ変数)を決める。

**結論:** 欠損は到着遅延の0.04%だけで、補完しない。カテゴリ変数4つはcategory型のままLightGBMに渡す。値ごとの満足率(Target Encoding)などのエンコーディングは、比べた結果を3節の表にまとめた。

In [1]:
import sys

sys.path.insert(0, "../src")  # 共通処理(features.py・modeling.py・public_oof.py)を読み込む
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import features
import modeling

plt.rcParams["font.family"] = "Meiryo"  # グラフの日本語ラベル用
DATA = features.DATA_DIR
train, test = features.load_train_and_test()
y = train[features.TARGET].astype(int).values
print(f"train {len(train):,}行・test {len(test):,}行、満足率 {y.mean():.3f}")

train 699,635行・test 299,844行、満足率 0.444


## 1. 欠損値の扱い

#### ① 目的・仮説
- **確認したいこと:** 欠損がある特徴量はどれか。補完する必要があるか。
- **なぜ必要か:** 欠損が多ければ、補完の仕方でスコアが変わりうる。補完するかどうかは、学習の前に決めておく必要がある。
- **想定する結果:** 人工データなので、欠損は少なく、補完しなくてもよい。

#### ② 検証方法
- **データ:** trainとtest。
- **方法:** 特徴量ごとに、欠損(NaN)の件数と割合を数える。

#### ③ コード・実行結果

In [2]:
missing = pd.DataFrame({"train": train.isna().sum(), "test": test.isna().sum()}).fillna(0).astype(int)
missing["trainの割合"] = (missing["train"] / len(train)).map("{:.2%}".format)
missing[missing[["train", "test"]].sum(axis=1) > 0]

,train,test,trainの割合
Arrival Delay in Minutes,292,130,0.04%


#### ④ 考察・次のアクション
- **読み取れること:** 欠損は到着遅延だけで、train 292行・test 130行(どちらも0.04%)。
- **想定との比較:** 想定どおり、欠損は少なかった。
- **次のアクション:**
  - 欠損は0.04%しかなく、補完してもAUCが上がる見込みは薄い。LightGBMは欠損をそのまま扱えるので、補完せずに進めた(00のベースラインも補完していない)。
  - 念のため、特徴量を作ったあとの土台で、欠損を同じ人の出発遅延で埋めて比べた(欠損していない行では、出発遅延と到着遅延の相関0.84、86.9%が完全に一致)。結果は3seed平均−0.000009でほぼ効果がなく、採用しなかった。
  - サービス評価の0点は、値は入っていても「評価なし」の可能性がある。これは01_EDAの7節で確かめた(0点の印は効かず、0点のまま渡す)。

## 2. カテゴリ変数の渡し方

#### ① 目的・仮説
- **確認したいこと:** 文字列のカテゴリ変数4つ(性別・顧客区分・旅行目的・座席クラス)を、どの形でモデルに渡すか。trainにない値がtestにあるか。
- **なぜ必要か:** 4つは中身はカテゴリ変数だが、CSVを読み込んだ時点の型は文字列(`str`)になる(CSVには型の情報が無いため)。LightGBMは文字列の列を受け取れないので、category型に変える必要がある。testにだけある値があると、予測のときに扱いが変わる。
- **想定する結果:** どれも値の種類が2〜3個と少なく、category型で渡せば足りる。trainとtestで値はそろっている。

#### ② 検証方法
- **データ:** trainとtest。
- **方法:** 読み込んだ時点の型を確かめる。カテゴリ変数ごとに、値の種類と件数をtrainとtestで並べる。`features.build_train_and_test_tables` でcategory型にしたあと、testのカテゴリ番号がtrainとそろっているかを確かめる。

#### ③ コード・実行結果

In [3]:
print("読み込んだ時点の型:", train[features.CATEGORICAL_COLUMNS].dtypes.astype(str).to_dict())
for c in features.CATEGORICAL_COLUMNS:
    counts = pd.DataFrame({"train": train[c].value_counts(), "test": test[c].value_counts()})
    print(f"{c}: {len(counts)}種類 / testにだけある値: {sorted(set(test[c].dropna()) - set(train[c].dropna()))}")
    print(counts.to_string(), "\n")

X, X_test = features.build_train_and_test_tables(train, test, feature_names=[])  # 元の特徴量20個だけ
category_columns = list(X.columns[X.dtypes == "category"])
print("category型にした特徴量:", category_columns)
print("testのカテゴリ番号がtrainと同じ:", all(list(X[c].cat.categories) == list(X_test[c].cat.categories) for c in category_columns))

読み込んだ時点の型: {'Gender': 'str', 'Customer Type': 'str', 'Type of Travel': 'str', 'Class': 'str'}


Gender: 2種類 / testにだけある値: []
         train    test
Gender                
Male    351683  150261
Female  347952  149583 



Customer Type: 2種類 / testにだけある値: []
                    train    test
Customer Type                    
Loyal Customer     576990  247708
disloyal Customer  122645   52136 



Type of Travel: 2種類 / testにだけある値: []
                  train    test
Type of Travel                 
Business travel  497441  213141
Personal Travel  202194   86703 



Class: 3種類 / testにだけある値: []
           train    test
Class                   
Business  342212  146626
Eco       327404  140220
Eco Plus   30019   12998 

category型にした特徴量: ['Class', 'Type of Travel', 'Customer Type']
testのカテゴリ番号がtrainと同じ: True


#### ④ 考察・次のアクション
- **読み取れること:** 読み込んだ時点の型は4つとも文字列(`str`)。4つとも値は2〜3種類で、testにだけある値はない。category型にしたあと、testのカテゴリ番号はtrainとそろっている(性別はベースラインで外したので、表には入らない)。
- **想定との比較:** 想定どおりだった。
- **次のアクション:** category型のままLightGBMに渡す。LightGBMはカテゴリの値を満足率の順に並べて分けるので、2〜3種類ならOne-Hotにしても分け方はほぼ同じになる。そのためOne-Hotは試さなかった。

## 3. エンコーディングの比較

カテゴリや値の渡し方(エンコーディング)は、ベースラインのあとに1つずつ比べて決めた。値ごとの満足率・出現回数は新しい特徴量として足したので、詳しい作り方は03_feature_engineering、CatBoostは04_modelingに書いた。

| 比べたもの | 結果 | 採否 |
|---|---|---|
| 飛行距離: 値そのまま vs 値ごとの満足率(Target Encoding) | 値ごとの満足率で+0.00136(1seed。特徴量エンジニアリングで最大の改善) | 採用 |
| 値ごとの満足率の強さ(人数の少ない値を全体の満足率に寄せる強さm=1・3・5・10・20・50) | m=3が最良(OOF 0.959983)。m=50は0.959752に下がった | m=3を採用 |
| 値の出現回数(Frequency Encoding)の13候補 | 飛行距離×旅行目的だけが効いた(+0.000057、3seed平均) | 1個だけ採用 |
| 飛行距離をcategory型(3,474種類)で渡す | −0.00121 | 不採用 |
| CatBoost: 一部の特徴量だけカテゴリ扱い vs 全部をカテゴリ扱い | 全部をカテゴリ扱いで+0.000187 | 採用 |
| カテゴリ4つのOne-Hot | 試していない(2節の理由) | — |

## Summary

| 決めたこと | 値 |
|---|---|
| 欠損値 | 補完しない(到着遅延の0.04%だけ。出発遅延で埋めても−0.000009) |
| カテゴリ変数 | category型のまま渡す(testのカテゴリ番号はtrainにそろえる) |
| エンコーディング | 飛行距離の値ごとの満足率(m=3)・飛行距離×旅行目的の出現回数を採用(03で特徴量として作る) |